<a href="https://colab.research.google.com/github/Eton-np/DataMining/blob/main/Chapter_2_Worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ใบงานปฏิบัติการ สัปดาห์ที่ 2
## 128-445 การทำคลังข้อมูลและเหมืองข้อมูล
### Getting to Know Your Data · Data Warehousing & OLAP

**ชื่อ–สกุล:** ................................................  **รหัสนักศึกษา:** ....................  **กลุ่ม:** ..........

---

### สิ่งที่จะได้ทำในใบงานนี้

| ปฏิบัติการ | หัวข้อ | เวลา |
|---|---|---|
| **1** | รู้จักข้อมูลของคุณ — สถิติเชิงพรรณนา Boxplot และการวัดความคล้าย | 25 นาที |
| **2** | สร้าง Star Schema จากตารางแบน | 20 นาที |
| **3** | ปฏิบัติการ OLAP ทั้ง 5 ท่า และตอบคำถามธุรกิจ | 20 นาที |

### สิ่งที่ต้องทำมาก่อนเข้าคาบ

- ทบทวนสไลด์สัปดาห์ที่ 1 หัวข้อ Fact และ Dimension
- เปิด Colab ให้พร้อม และเตรียมไฟล์ `supermarket_sales.csv` ไว้เผื่อเครือข่ายบล็อก GitHub

### กติกา

1. **เขียนโค้ดเอง** ทุกช่องที่เขียนว่า `# TODO`
2. **ห้ามเดา** — ทุกคำตอบต้องมีโค้ดที่พิสูจน์ได้
3. ใช้ AI ช่วยได้ แต่ต้อง **รันเองและตรวจสอบผลเอง** และระบุไว้ท้ายใบงาน
4. เซลล์ที่ขึ้นต้นด้วย `ck(...)` คือเซลล์ตรวจคำตอบอัตโนมัติ

---
## ขั้นที่ 0 · เตรียมข้อมูล

รันเซลล์ข้างล่างนี้ก่อนเสมอ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = ('https://raw.githubusercontent.com/plotly/datasets/master/'
       'supermarket_Sales.csv')
try:
    df = pd.read_csv(URL)
    print("โหลดจากอินเทอร์เน็ตสำเร็จ")
except Exception:
    df = pd.read_csv('supermarket_sales.csv')
    print("โหลดจากไฟล์ในเครื่องสำเร็จ")

df.columns = (df.columns.str.strip().str.lower()
                .str.replace(' ', '_').str.replace('%', ''))
df = df.rename(columns={'customer_stratification_rating': 'rating',
                        'cost_of_goods_sold': 'cogs',
                        'gross_margin_percentage': 'gross_margin_pct'})
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')
df['hour'] = pd.to_datetime(df['time'], format='%H:%M').dt.hour
print(df.shape)
df.head(3)

In [ ]:
# ===== เซลล์ตรวจคำตอบ — ห้ามแก้ =====
import base64
_A = {"median": "MjUzLjg1", "skew": "MC44OTM=", "iqr": "MzQ2Ljkz", "upper_fence": "OTkxLjc0", "n_outliers": "OQ==", "cv_total": "NzYuMQ==", "cos_hs": "MC45OTA4", "dim_date": "ODk=", "fact_rows": "MTAwMA==", "fact_total": "MzIyOTY2Ljc1", "dow_gap": "NDguMQ==", "c_weakest": "MTM4OTUuNTU=", "dice_total": "NzI1NDQuNzc="}

def ck(key, value, hint=""):
    """ตรวจคำตอบ (เฉลยถูกเข้ารหัสไว้ เพื่อไม่ให้อ่านเจอโดยบังเอิญ)"""
    exp = base64.b64decode(_A[key]).decode()
    try:
        tol = max(0.02, abs(float(exp)) * 1e-5)
        ok = abs(float(value) - float(exp)) <= tol
    except (TypeError, ValueError):
        ok = str(value).strip() == exp
    print(("[ถูกต้อง]   " if ok else "[ยังไม่ถูก] ") + key + " = " + str(value)
          + ("" if ok else "   " + hint))
    return ok

print("โหลดตัวตรวจคำตอบเรียบร้อย")

---
# ปฏิบัติการที่ 1 · รู้จักข้อมูลของคุณ
**เวลา 25 นาที · ทำเป็นกลุ่ม 4 คน**

> เป้าหมาย: สรุปลักษณะของชุดข้อมูลด้วยสถิติที่ **เลือกถูกตัว** และอ่าน Boxplot ให้ออก

### 1.1 ระดับการวัดของแต่ละคอลัมน์

ก่อนคำนวณอะไรทั้งสิ้น ต้องรู้ก่อนว่าคอลัมน์นั้นคำนวณอะไรได้บ้าง

In [ ]:
# TODO: แสดงจำนวนค่าไม่ซ้ำของทุกคอลัมน์ (เรียงจากน้อยไปมาก) และชนิดข้อมูลของทุกคอลัมน์


**เขียนตอบ** — จัดประเภทคอลัมน์ตามระดับการวัด

| คอลัมน์ | Nominal / Ordinal / Interval / Ratio | คำนวณค่าเฉลี่ยได้ไหม |
|---|---|---|
| `branch` | | |
| `customer_type` | | |
| `date` | | |
| `total` | | |
| `rating` | | |

**คำถาม:** ทำไมการคำนวณค่าเฉลี่ยของ `branch` ถึงไม่มีความหมาย ทั้งที่ Python คำนวณให้ได้ถ้าแปลงเป็นตัวเลข

*ตอบ:* ...........................................................................................................

### 1.2 การวัดแนวโน้มสู่ส่วนกลาง — ค่าเฉลี่ยหลอกเราได้อย่างไร

In [ ]:
# TODO: คำนวณค่าเฉลี่ย มัธยฐาน และความเบ้ ของคอลัมน์ total
#       ปัด: ค่าเฉลี่ยและมัธยฐาน 2 ตำแหน่ง · ความเบ้ 3 ตำแหน่ง
s = df['total']
mean_total = ...
median_total = ...
skew_total = ...

print("ค่าเฉลี่ย  :", mean_total)
print("มัธยฐาน   :", median_total)
print("ผลต่าง    :", round(mean_total - median_total, 2))
print("ความเบ้   :", skew_total)

In [ ]:
ck("median", median_total, "ลองใช้ s.median()")
ck("skew", skew_total, "ลองใช้ s.skew() แล้วปัด 3 ตำแหน่ง")

**หยุดคิด:** ค่าเฉลี่ยสูงกว่ามัธยฐานอยู่ประมาณ 69 บาท และความเบ้เป็นบวก

> ถ้ารายงานผู้บริหารว่า *"ลูกค้าใช้จ่ายเฉลี่ย 323 บาทต่อบิล"* จะเกิดความเข้าใจผิดอย่างไร
>
> *ตอบ:* .........................................................................................................

**ลองเปรียบเทียบกับคอลัมน์ `rating`** — รันเซลล์ข้างล่าง แล้วสังเกตว่าต่างจาก `total` อย่างไร

In [ ]:
r = df['rating']
print("rating  mean %.2f  median %.2f  skew %.3f" % (r.mean(), r.median(), r.skew()))
print("total   mean %.2f  median %.2f  skew %.3f" % (s.mean(), s.median(), s.skew()))

### 1.3 Five-number Summary และค่าผิดปกติ

In [ ]:
# TODO: หา Q1, Q3, IQR, รั้วบน, รั้วล่าง และนับจำนวนค่าผิดปกติของ total
#       สูตร: IQR = Q3 - Q1 · รั้วบน = Q3 + 1.5*IQR · รั้วล่าง = Q1 - 1.5*IQR
#       ปัดทุกค่า 2 ตำแหน่ง
q1 = ...
q3 = ...
iqr = ...
upper_fence = ...
lower_fence = ...
n_outliers = ...

print("ต่ำสุด %.2f | Q1 %.2f | มัธยฐาน %.2f | Q3 %.2f | สูงสุด %.2f"
      % (s.min(), q1, s.median(), q3, s.max()))
print("IQR       :", iqr)
print("รั้วล่าง   :", lower_fence)
print("รั้วบน    :", upper_fence)
print("ค่าผิดปกติ:", n_outliers, "รายการ")

In [ ]:
ck("iqr", iqr, "IQR = Q3 - Q1")
ck("upper_fence", upper_fence, "รั้วบน = Q3 + 1.5 * IQR")
ck("n_outliers", n_outliers, "นับแถวที่ total เกินรั้วบน หรือ ต่ำกว่ารั้วล่าง")

**คำถาม:** รั้วล่างที่คำนวณได้ติดลบ แปลว่าอะไรในบริบทของยอดขาย

*ตอบ:* ...........................................................................................................

### 1.4 วาด Boxplot และ Histogram

รันได้เลย ไม่ต้องแก้ — แล้วตอบคำถามใต้กราฟ

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

bp = ax[0].boxplot(s, vert=False, widths=0.55, patch_artist=True,
                   flierprops=dict(marker='o', markersize=4,
                                   markerfacecolor='#D95F45',
                                   markeredgecolor='#D95F45'))
bp['boxes'][0].set(facecolor='#E8EFF7', edgecolor='#2E77B8', linewidth=1.6)
bp['medians'][0].set(color='#D95F45', linewidth=2.2)
for w in bp['whiskers'] + bp['caps']:
    w.set(color='#16375E', linewidth=1.6)
ax[0].set_title('Boxplot of total', loc='left', fontsize=11)
ax[0].set_yticks([])

ax[1].hist(s, bins=20, color='#2E77B8', edgecolor='white', linewidth=0.8)
ax[1].axvline(s.mean(), color='#E8A33D', linewidth=2, label='mean')
ax[1].axvline(s.median(), color='#2E9E6B', linewidth=2, label='median')
ax[1].legend(frameon=False)
ax[1].set_title('Histogram of total', loc='left', fontsize=11)

for a in ax:
    a.spines[['top', 'right']].set_visible(False)
    a.grid(axis='x', color='#D8E1EA', linewidth=0.8)
    a.set_axisbelow(True)
plt.tight_layout(); plt.show()

**อ่านกราฟแล้วตอบ**

1. มัธยฐาน (เส้นสีส้ม) อยู่กลางกล่องพอดีหรือค่อนไปทางใด และบอกอะไรเกี่ยวกับรูปร่างการแจกแจง

   *ตอบ:* ..............................................................................................

2. ในฮิสโทแกรม เส้นค่าเฉลี่ยกับเส้นมัธยฐานอยู่ตรงไหนเทียบกับยอดบิลที่หนาแน่นที่สุด

   *ตอบ:* ..............................................................................................

### 1.5 การกระจาย — เทียบข้ามคอลัมน์ที่หน่วยต่างกัน

ส่วนเบี่ยงเบนมาตรฐานเทียบข้ามคอลัมน์ไม่ได้ เพราะหน่วยต่างกัน จึงต้องใช้ **สัมประสิทธิ์การแปรผัน (CV)**

```
CV = (ส่วนเบี่ยงเบนมาตรฐาน ÷ ค่าเฉลี่ย) × 100
```

In [ ]:
# TODO: คำนวณ CV ของ total, unit_price และ rating ปัดทศนิยม 1 ตำแหน่ง
cv_total = ...
cv_price = ...
cv_rating = ...

print("CV ของ total      : %.1f %%" % cv_total)
print("CV ของ unit_price : %.1f %%" % cv_price)
print("CV ของ rating     : %.1f %%" % cv_rating)

In [ ]:
ck("cv_total", cv_total, "CV = std / mean * 100")

**คำถาม:** คอลัมน์ใดกระจายมากที่สุด และเรื่องนี้จะสำคัญอย่างไรตอนที่เราต้องปรับสเกลข้อมูลก่อนใช้อัลกอริทึมที่อาศัยระยะทาง

*ตอบ:* ...........................................................................................................

### 1.6 การวัดความคล้าย — กลุ่มสินค้าใดมีพฤติกรรมคล้ายกัน

แทนกลุ่มสินค้าแต่ละกลุ่มด้วยเวกเตอร์ยอดขาย 3 มิติ (สาขา A, B, C) แล้ววัดความคล้ายสองแบบ

In [ ]:
# TODO: สร้าง pivot กลุ่มสินค้า (แถว) x สาขา (คอลัมน์) ด้วยผลรวมของ total
#       เก็บไว้ในตัวแปร p แล้วปัดเป็นจำนวนเต็ม
p = ...

print(p)

In [ ]:
# TODO: เขียนฟังก์ชัน euclidean(x, y) และ cosine(x, y) เอง (ห้ามใช้ไลบรารีสำเร็จรูป)
#       สูตร Euclidean : sqrt(sum((x - y)**2))
#       สูตร Cosine    : (x . y) / (|x| * |y|)      คำใบ้: x @ y และ np.linalg.norm(x)
def euclidean(x, y):
    ...

def cosine(x, y):
    ...

home = p.loc['Home and lifestyle'].values.astype(float)
sports = p.loc['Sports and travel'].values.astype(float)
food = p.loc['Food and beverages'].values.astype(float)

cos_hs = ...
cos_hf = ...

print("Euclidean(Home, Sports) = %.0f" % euclidean(home, sports))
print("Euclidean(Home, Food)   = %.0f" % euclidean(home, food))
print("Cosine(Home, Sports)    = %.4f" % cos_hs)
print("Cosine(Home, Food)      = %.4f" % cos_hf)

In [ ]:
ck("cos_hs", cos_hs, "Cosine = (x @ y) / (norm(x) * norm(y)) แล้วปัด 4 ตำแหน่ง")

**อภิปรายในกลุ่ม**

1. ทั้งสองมาตรวัดชี้ไปที่คำตอบเดียวกันหรือไม่ ว่ากลุ่มใดคล้าย `Home and lifestyle` มากที่สุด

   *ตอบ:* ..............................................................................................

2. ถ้าคุณเป็นผู้จัดการที่ต้องจัดผังร้าน ควรใช้มาตรวัดใด และเพราะอะไร

   *ตอบ:* ..............................................................................................

> **สิ่งที่ต้องจำ:** Euclidean บอกว่า *ห่างกันเท่าไร* · Cosine บอกว่า *ไปในทิศทางเดียวกันหรือไม่*
> เลือกให้ตรงกับคำถามทางธุรกิจ ไม่ใช่เลือกตามความเคยชิน

---
# ปฏิบัติการที่ 2 · สร้าง Star Schema
**เวลา 20 นาที · ทำเป็นกลุ่ม 4 คน**

> เป้าหมาย: แปลงตารางแบน 1,000 แถว 17 คอลัมน์ ให้เป็นตาราง Fact 1 ตาราง และ Dimension 4 ตาราง
> แล้วพิสูจน์ว่า **ไม่มีข้อมูลสูญหายไปแม้แต่บาทเดียว**

### 2.1 ออกแบบก่อนเขียนโค้ด

ก่อนแตะคีย์บอร์ด ให้กลุ่มตอบคำถามนี้ในใบงานหน้า 1 ก่อน

- **Grain** ของตาราง Fact คืออะไร (หนึ่งแถวแทนอะไร)
- คอลัมน์ใดเป็น **ตัววัด (Fact)** และคอลัมน์ใดเป็น **มิติ (Dimension)**
- ตาราง Dimension แต่ละตารางควรมีกี่แถว — **เดาก่อน** แล้วค่อยพิสูจน์

### 2.2 สร้างตาราง Dimension

In [ ]:
# TODO: สร้างตาราง Dimension 4 ตาราง โดยใช้ drop_duplicates() แล้ว reset_index(drop=True)
#       dim_branch   จากคอลัมน์ branch, city
#       dim_product  จากคอลัมน์ product_line
#       dim_customer จากคอลัมน์ customer_type, gender
#       dim_date     จากคอลัมน์ date
dim_branch = ...
dim_product = ...
dim_customer = ...
dim_date = ...

# TODO: เพิ่มคอลัมน์คีย์หลักให้ทุกตาราง โดยใช้ .index
#       เช่น dim_branch['branch_key'] = dim_branch.index


for name, d in [('dim_branch', dim_branch), ('dim_product', dim_product),
                ('dim_customer', dim_customer), ('dim_date', dim_date)]:
    print("%-14s %4d แถว" % (name, len(d)))

In [ ]:
ck("dim_date", len(dim_date), "นับวันที่ไม่ซ้ำ — ควรน้อยกว่า 90 วัน")

**หยุดคิด:** `dim_date` มี 89 แถว ทั้งที่ช่วง ม.ค. ถึง มี.ค. 2019 มี 90 วัน

> แปลว่าอะไร และคุณจะหาว่าเป็นวันไหนได้อย่างไร (เขียนโค้ดในเซลล์ถัดไปถ้าเหลือเวลา)
>
> *ตอบ:* .........................................................................................................

### 2.3 สร้างตาราง Fact

In [ ]:
# TODO: สร้างตาราง fact โดย merge df เข้ากับ Dimension ทั้ง 4 ตาราง
#       แล้วเก็บเฉพาะคอลัมน์: invoice_id, คีย์นอก 4 ตัว, quantity, unit_price, total, gross_income
fact = ...

fact_rows = len(fact)
print("fact_sales มี", fact_rows, "แถว")
fact.head(3)

In [ ]:
ck("fact_rows", fact_rows, "ถ้าไม่เท่ากับ 1000 แปลว่า merge ผิด — อาจเกิดแถวซ้ำหรือแถวหาย")

### 2.4 ตรวจสอบความถูกต้อง — ข้อมูลหายไปไหม

นี่คือขั้นตอนที่ **ห้ามข้าม** ในงานจริง

In [ ]:
# TODO: ตรวจว่ายอดรวมของ total ในตาราง Fact เท่ากับในตารางเดิมหรือไม่
#       และตรวจว่า invoice_id ในตาราง Fact ไม่ซ้ำกันเลย
fact_total = ...
orig_total = ...

print("ยอดรวมในตารางเดิม :", orig_total)
print("ยอดรวมในตาราง Fact:", fact_total)
print("ตรงกันหรือไม่     :", np.isclose(fact_total, orig_total))
print("จำนวนบิลไม่ซ้ำ    :", fact['invoice_id'].nunique())

In [ ]:
ck("fact_total", fact_total, "ถ้าไม่ตรงกับตารางเดิม แปลว่า merge ทำให้แถวซ้ำหรือหาย")

### 2.5 ประหยัดพื้นที่ได้เท่าไร

รันได้เลย — ตัวเลขอาจต่างกันเล็กน้อยตามเวอร์ชัน pandas แต่แนวโน้มจะเหมือนกัน

In [ ]:
flat_kb = df.memory_usage(deep=True).sum() / 1024
star_kb = (fact.memory_usage(deep=True).sum()
           + dim_branch.memory_usage(deep=True).sum()
           + dim_product.memory_usage(deep=True).sum()
           + dim_customer.memory_usage(deep=True).sum()
           + dim_date.memory_usage(deep=True).sum()) / 1024

print("ตารางแบน   : %8.1f KB" % flat_kb)
print("Star Schema: %8.1f KB" % star_kb)
print("ประหยัดได้  : %8.1f %%" % ((1 - star_kb / flat_kb) * 100))

**อภิปรายในกลุ่ม**

1. พื้นที่ที่ประหยัดได้มาจากไหน (คิดถึงคำว่า `Naypyitaw` ที่เคยถูกเก็บซ้ำกี่ครั้ง)

   *ตอบ:* ..............................................................................................

2. ถ้าเป็นห้าง 500 สาขา ขายวันละ 20,000 บิล เป็นเวลา 5 ปี ตาราง Fact จะมีกี่แถว และการประหยัดนี้จะสำคัญขึ้นแค่ไหน

   *ตอบ:* ..............................................................................................

---
# ปฏิบัติการที่ 3 · ปฏิบัติการ OLAP ทั้ง 5 ท่า
**เวลา 20 นาที · ทำเป็นกลุ่ม 4 คน**

> เป้าหมาย: ใช้ท่า OLAP ทั้ง 5 ท่ากับข้อมูลจริง แล้วปิดท้ายด้วย **ข้อเสนอแนะเชิงธุรกิจ 1 ข้อ**

| ท่า | ทำอะไร | คำสั่ง pandas |
|---|---|---|
| Roll-up | สรุปขึ้นระดับหยาบกว่า | `groupby()` ที่มิติหยาบ |
| Drill-down | เจาะลงระดับละเอียดกว่า | `groupby()` ที่มิติละเอียด |
| Slice | เลือกค่าเดียวในหนึ่งมิติ | การกรองด้วยเงื่อนไข |
| Dice | เลือกช่วงค่าในหลายมิติ | เงื่อนไขหลายตัวรวมกัน |
| Pivot | หมุนแกนเปลี่ยนมุมมอง | `pivot_table()` |

### 3.1 Roll-up — ยอดขายรายเดือน

In [ ]:
# TODO: สร้างคอลัมน์ month จาก date แล้วหายอดขายรวมรายเดือน
#       คำใบ้: df['date'].dt.to_period('M').astype(str)
df['month'] = ...
by_month = ...

print(by_month)

**กับดักที่ต้องระวัง:** กุมภาพันธ์มียอดต่ำที่สุด แต่กุมภาพันธ์มีเพียง 28 วัน

รันเซลล์ข้างล่างเพื่อเทียบอย่างเป็นธรรม

In [ ]:
days = {'2019-01': 31, '2019-02': 28, '2019-03': 31}
per_day = (by_month / pd.Series(days)).round(2)
print("ยอดขายเฉลี่ยต่อวัน")
print(per_day)

**คำถาม:** เมื่อเทียบเป็นยอดต่อวันแล้ว กุมภาพันธ์ยังต่ำที่สุดอยู่ไหม และช่องว่างแคบลงแค่ไหน

*ตอบ:* ...........................................................................................................

### 3.2 Drill-down — เจาะลงถึงวันในสัปดาห์

In [ ]:
# TODO: สร้างคอลัมน์ dow (ชื่อวันในสัปดาห์) แล้วหายอดขายรวมของแต่ละวัน
#       เรียงจากมากไปน้อย  คำใบ้: df['date'].dt.day_name()
df['dow'] = ...
by_dow = ...

# TODO: คำนวณว่าวันที่ขายดีที่สุด สูงกว่าวันที่แย่ที่สุดกี่เปอร์เซ็นต์ ปัด 1 ตำแหน่ง
dow_gap = ...

print(by_dow)
print("วันที่ขายดีที่สุดสูงกว่าวันที่แย่ที่สุด %.1f %%" % dow_gap)

In [ ]:
ck("dow_gap", dow_gap, "(สูงสุด / ต่ำสุด - 1) * 100")

> **สังเกต:** ข้อมูลระดับเดือนมองไม่เห็นรูปแบบรายสัปดาห์นี้เลย — นี่คือคุณค่าที่แท้จริงของการ Drill-down

### 3.3 Slice — ดูเฉพาะสาขา C

In [ ]:
# TODO: Slice เอาเฉพาะสาขา C (city == 'Naypyitaw')
#       แล้วหายอดขายรวมของแต่ละกลุ่มสินค้า เรียงจากน้อยไปมาก
branch_c = ...
c_by_product = ...

# TODO: เก็บยอดของกลุ่มที่อ่อนที่สุดไว้ในตัวแปร c_weakest (ปัด 2 ตำแหน่ง)
c_weakest = ...

print(c_by_product)
print("กลุ่มที่อ่อนที่สุดของสาขา C คือ", c_by_product.index[0], "=", c_weakest)

In [ ]:
ck("c_weakest", c_weakest, "กรองด้วย city == 'Naypyitaw' ก่อน แล้วค่อย groupby")

### 3.4 Dice — หั่นหลายมิติพร้อมกัน

**โจทย์:** ยอดขายของ **สาขา A และ B** เฉพาะกลุ่ม **Home and lifestyle และ Health and beauty**

In [ ]:
# TODO: สร้างเงื่อนไข mask ที่เลือก สาขา A หรือ B
#       และกลุ่มสินค้า Home and lifestyle หรือ Health and beauty
#       คำใบ้: ใช้ .isin([...]) และเชื่อมด้วย &
mask = ...
diced = df[mask]
dice_total = ...

print("จำนวนบิลที่เข้าเงื่อนไข :", len(diced))
print("ยอดขายรวม             :", dice_total)

In [ ]:
ck("dice_total", dice_total, "อย่าลืมใส่วงเล็บครอบแต่ละเงื่อนไขก่อนเชื่อมด้วย &")

### 3.5 Pivot — หมุนแกนเพื่อเปลี่ยนมุมมอง

คำถาม: **แต่ละสาขาขายดีที่สุดคนละชั่วโมงกันหรือไม่**

In [ ]:
# TODO: สร้าง pivot ที่มี แถว = hour, คอลัมน์ = branch, ค่า = ผลรวมของ total
hour_branch = ...

print(hour_branch)
print("ชั่วโมงที่ขายดีที่สุดของแต่ละสาขา:")
print(hour_branch.idxmax())

**คำถาม:** ทั้ง 3 สาขาขายดีที่สุดในชั่วโมงเดียวกันหรือไม่ ถ้าไม่ สาขาใดต่างออกไป และคุณจะตั้งสมมติฐานว่าอะไร

*ตอบ:* ...........................................................................................................

> ถ้าดูแค่ยอดรวมทั้งห้าง จะเห็นแค่ว่า 19:00 น. ขายดีที่สุด — แต่พอ Pivot แล้วจะเห็นว่าไม่จริงกับทุกสาขา

### 3.6 สรุปเป็นข้อเสนอแนะเชิงธุรกิจ

ทำในใบงานหน้า 4 — เขียนข้อเสนอแนะ 1 ข้อที่ **ผู้จัดการนำไปทำได้จริงในสัปดาห์หน้า** โดยต้องอ้างตัวเลขจากปฏิบัติการนี้อย่างน้อย 2 ตัว

---
# สรุปท้ายใบงาน

### เขียนตอบ 3 ข้อ

1. ท่า OLAP ท่าใดที่คุณคิดว่า **มีประโยชน์ที่สุด** กับข้อมูลชุดนี้ และเพราะอะไร

   *ตอบ:* ..............................................................................................

2. สิ่งที่ค้นพบวันนี้ที่ **ยอดรวมทั้งห้างมองไม่เห็น** คืออะไร ยกมา 1 ตัวอย่างพร้อมตัวเลข

   *ตอบ:* ..............................................................................................

3. ถ้าจะเพิ่มมิติใหม่เข้าไปใน Star Schema นี้อีก 1 มิติ คุณจะเพิ่มมิติอะไร และจะใช้ตอบคำถามอะไร

   *ตอบ:* ..............................................................................................

---

### การใช้ AI ในใบงานนี้ (ตอบตามจริง)

| ข้อ | คำตอบ |
|---|---|
| ใช้เครื่องมือ AI ใดบ้าง | .............................................. |
| ใช้ช่วยในส่วนใด | .............................................. |
| ตรวจสอบผลลัพธ์อย่างไร | .............................................. |
| ไม่ได้ใช้เลย | ☐ |

---

### รายการตรวจก่อนส่ง

- [ ] รันทุกเซลล์ตั้งแต่ต้นจนจบแล้ว ไม่มี error
- [ ] เซลล์ `ck(...)` ทุกอันขึ้นว่า **[ถูกต้อง]**
- [ ] ตอบคำถามเขียนตอบครบทุกข้อ
- [ ] เขียนข้อเสนอแนะเชิงธุรกิจในใบงานหน้า 4 แล้ว
- [ ] เปลี่ยนชื่อไฟล์เป็น `W2_รหัสนักศึกษา_ชื่อ.ipynb` แล้วส่งใน Google Classroom

**กำหนดส่ง:** วันอาทิตย์ เวลา 23.59 น.